In [3]:
from pathlib import Path
import os
import sys
from IPython.display import display, Markdown

WORKSPACE = Path(
    r"C:\Users\menna\Documents\Codex\2026-11-19\https-github-com-sandyreda898-oss-smart"
)
ML_ROOT = WORKSPACE / "outputs" / "machine_learning"

required_paths = [
    ML_ROOT / "src" / "model_loader.py",
    ML_ROOT / "models",
    ML_ROOT / "reports",
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required project path was not found: {path}")

os.chdir(ML_ROOT)
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.model_loader import load_model, ModelUnavailableError

try:
    model, metadata, metrics, schema = load_model(ML_ROOT)

except ModelUnavailableError as exc:
    message = (
        "# Explainability report\n\n"
        "**Status: blocked - no released model exists.**\n\n"
        f"Reason: {exc}\n\n"
        "Feature importance and SHAP are not meaningful without a model "
        "trained on verified measurements.\n"
    )

    report_path = ML_ROOT / "reports" / "explainability_report.md"
    report_path.write_text(message, encoding="utf-8")

    print("Explainability report:", report_path)
    display(Markdown(message))

else:
    import pandas as pd
    import shap
    import matplotlib.pyplot as plt

    dataset = pd.read_csv(
        ML_ROOT / "data" / "processed" / "training_dataset.csv",
        low_memory=False,
    )

    features = schema["features"]
    sample = dataset[features].sample(
        min(200, len(dataset)),
        random_state=42,
    )

    transformed = model.named_steps["preprocessing"].transform(sample)
    names = model.named_steps["preprocessing"].get_feature_names_out()

    explainer = shap.Explainer(
        model.named_steps["model"],
        transformed,
        feature_names=names,
    )
    explanation = explainer(transformed)

    output_dir = ML_ROOT / "visualizations" / "explainability"
    output_dir.mkdir(parents=True, exist_ok=True)

    shap.plots.beeswarm(explanation, show=False)
    plt.tight_layout()

    plot_path = output_dir / "shap_summary.png"
    plt.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.close()

    print("SHAP summary plot:", plot_path)

Explainability report: C:\Users\menna\Documents\Codex\2026-11-19\https-github-com-sandyreda898-oss-smart\outputs\machine_learning\reports\explainability_report.md


# Explainability report

**Status: blocked - no released model exists.**

Reason: No complete model artifact set is available. Training may be blocked until verified field measurements are supplied.

Feature importance and SHAP are not meaningful without a model trained on verified measurements.
